# Module 2: Training & Optimization

This notebook covers the training process for neural networks.

**Topics covered:**
- Loss functions
- Backpropagation
- Gradient descent variants
- Regularization techniques

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Callable

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 2.1 Loss Functions

In [ ]:
def mse_loss(y_pred, y_true):
    """Mean Squared Error for regression."""
    return np.mean((y_pred - y_true) ** 2)

def mse_gradient(y_pred, y_true):
    """Gradient of MSE w.r.t. predictions."""
    return 2 * (y_pred - y_true) / len(y_true)

def softmax(z):
    """Softmax with numerical stability."""
    z = z - np.max(z, axis=-1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=-1, keepdims=True)

def cross_entropy_loss(logits, labels):
    """Cross-entropy loss for classification."""
    probs = softmax(logits)
    n = len(labels)
    return -np.mean(np.log(probs[np.arange(n), labels] + 1e-10))

def cross_entropy_gradient(logits, labels):
    """Gradient of cross-entropy w.r.t. logits."""
    probs = softmax(logits)
    grad = probs.copy()
    grad[np.arange(len(labels)), labels] -= 1
    return grad / len(labels)

In [ ]:
# Visualize cross-entropy loss
p = np.linspace(0.01, 0.99, 100)
loss = -np.log(p)

plt.figure(figsize=(10, 6))
plt.plot(p, loss, 'b-', linewidth=2)
plt.xlabel('Predicted Probability for True Class', fontsize=12)
plt.ylabel('Cross-Entropy Loss', fontsize=12)
plt.title('Cross-Entropy Loss: Penalizes Low Confidence', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

## 2.2 Backpropagation

The chain rule in action.

In [ ]:
class Layer:
    """Base class for layers with forward/backward."""
    def forward(self, x):
        raise NotImplementedError
    def backward(self, grad):
        raise NotImplementedError

class Linear(Layer):
    """Linear layer with backprop."""
    def __init__(self, n_in, n_out):
        self.W = np.random.randn(n_in, n_out) * np.sqrt(2.0 / n_in)
        self.b = np.zeros(n_out)
        self.x = None
        self.grad_W = None
        self.grad_b = None
    
    def forward(self, x):
        self.x = x
        return x @ self.W + self.b
    
    def backward(self, grad):
        self.grad_W = self.x.T @ grad
        self.grad_b = np.sum(grad, axis=0)
        return grad @ self.W.T

class ReLU(Layer):
    """ReLU activation with backprop."""
    def __init__(self):
        self.mask = None
    
    def forward(self, x):
        self.mask = x > 0
        return np.maximum(0, x)
    
    def backward(self, grad):
        return grad * self.mask

In [ ]:
# Simple network with backprop
class SimpleNetwork:
    def __init__(self, sizes):
        self.layers = []
        for i in range(len(sizes) - 1):
            self.layers.append(Linear(sizes[i], sizes[i+1]))
            if i < len(sizes) - 2:  # No activation after last layer
                self.layers.append(ReLU())
    
    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x
    
    def backward(self, grad):
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
    
    def get_params_and_grads(self):
        params, grads = [], []
        for layer in self.layers:
            if isinstance(layer, Linear):
                params.extend([layer.W, layer.b])
                grads.extend([layer.grad_W, layer.grad_b])
        return params, grads

# Test
net = SimpleNetwork([2, 4, 3])
X = np.random.randn(10, 2)
y = np.random.randint(0, 3, 10)

logits = net.forward(X)
loss = cross_entropy_loss(logits, y)
grad = cross_entropy_gradient(logits, y)
net.backward(grad)

print(f"Loss: {loss:.4f}")
params, grads = net.get_params_and_grads()
print(f"Number of parameter arrays: {len(params)}")

## 2.3 Gradient Descent Optimizers

In [ ]:
class SGD:
    """Vanilla SGD."""
    def __init__(self, params, lr=0.01):
        self.params = params
        self.lr = lr
    
    def step(self, grads):
        for p, g in zip(self.params, grads):
            p -= self.lr * g

class SGDMomentum:
    """SGD with momentum."""
    def __init__(self, params, lr=0.01, momentum=0.9):
        self.params = params
        self.lr = lr
        self.momentum = momentum
        self.velocities = [np.zeros_like(p) for p in params]
    
    def step(self, grads):
        for i, (p, g) in enumerate(zip(self.params, grads)):
            self.velocities[i] = self.momentum * self.velocities[i] + g
            p -= self.lr * self.velocities[i]

class Adam:
    """Adam optimizer."""
    def __init__(self, params, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = params
        self.lr = lr
        self.beta1, self.beta2 = beta1, beta2
        self.eps = eps
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0
    
    def step(self, grads):
        self.t += 1
        for i, (p, g) in enumerate(zip(self.params, grads)):
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g**2
            m_hat = self.m[i] / (1 - self.beta1**self.t)
            v_hat = self.v[i] / (1 - self.beta2**self.t)
            p -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

In [ ]:
# Compare optimizers on a simple 2D problem
def rosenbrock(x, y):
    return (1 - x)**2 + 100 * (y - x**2)**2

def rosenbrock_grad(x, y):
    dx = -2*(1 - x) - 400*x*(y - x**2)
    dy = 200*(y - x**2)
    return np.array([dx, dy])

def optimize(optimizer_class, lr, steps=500, **kwargs):
    point = np.array([-1.0, 1.0])
    params = [point]
    opt = optimizer_class(params, lr=lr, **kwargs)
    history = [point.copy()]
    
    for _ in range(steps):
        grad = rosenbrock_grad(point[0], point[1])
        opt.step([grad])
        history.append(point.copy())
    
    return np.array(history)

# Run optimizers
sgd_path = optimize(SGD, lr=0.0001, steps=1000)
momentum_path = optimize(SGDMomentum, lr=0.0001, momentum=0.9, steps=1000)
adam_path = optimize(Adam, lr=0.01, steps=1000)

In [ ]:
# Plot optimization paths
fig, ax = plt.subplots(figsize=(12, 8))

# Contour plot
x = np.linspace(-2, 2, 100)
y = np.linspace(-1, 3, 100)
X, Y = np.meshgrid(x, y)
Z = rosenbrock(X, Y)
ax.contour(X, Y, np.log(Z + 1), levels=30, cmap='viridis', alpha=0.5)

# Paths
ax.plot(sgd_path[:, 0], sgd_path[:, 1], 'r-', label='SGD', linewidth=2)
ax.plot(momentum_path[:, 0], momentum_path[:, 1], 'b-', label='SGD+Momentum', linewidth=2)
ax.plot(adam_path[:, 0], adam_path[:, 1], 'g-', label='Adam', linewidth=2)

ax.scatter([1], [1], c='gold', s=200, marker='*', zorder=5, label='Optimum')
ax.scatter([-1], [1], c='black', s=100, marker='o', zorder=5, label='Start')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Optimizer Comparison on Rosenbrock Function')
ax.legend()
plt.show()

## 2.4 Regularization

In [ ]:
def l2_regularization(params, lambda_reg):
    """Compute L2 regularization loss and gradients."""
    reg_loss = 0
    reg_grads = []
    for p in params:
        reg_loss += lambda_reg * np.sum(p ** 2)
        reg_grads.append(2 * lambda_reg * p)
    return reg_loss, reg_grads

def dropout_forward(x, p=0.5, training=True):
    """Apply dropout during training."""
    if not training:
        return x, None
    mask = np.random.rand(*x.shape) > p
    return x * mask / (1 - p), mask

def dropout_backward(grad, mask, p=0.5):
    """Backward pass for dropout."""
    return grad * mask / (1 - p)

In [ ]:
# Demonstrate dropout
x = np.ones((1, 10))
print("Original:", x)

for p in [0.0, 0.3, 0.5, 0.7]:
    dropped, mask = dropout_forward(x, p=p, training=True)
    print(f"Dropout p={p}: {dropped.round(2)} (mean={dropped.mean():.2f})")

## 2.5 Training Loop Example

In [ ]:
# Generate synthetic data
from sklearn.datasets import make_moons
X_train, y_train = make_moons(n_samples=500, noise=0.2, random_state=42)
X_test, y_test = make_moons(n_samples=100, noise=0.2, random_state=43)

# Normalize
mean, std = X_train.mean(axis=0), X_train.std(axis=0)
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

In [ ]:
def train_epoch(net, X, y, optimizer, batch_size=32):
    """Train for one epoch."""
    n = len(X)
    indices = np.random.permutation(n)
    total_loss = 0
    
    for i in range(0, n, batch_size):
        batch_idx = indices[i:i+batch_size]
        X_batch, y_batch = X[batch_idx], y[batch_idx]
        
        # Forward
        logits = net.forward(X_batch)
        loss = cross_entropy_loss(logits, y_batch)
        total_loss += loss * len(batch_idx)
        
        # Backward
        grad = cross_entropy_gradient(logits, y_batch)
        net.backward(grad)
        
        # Update
        params, grads = net.get_params_and_grads()
        optimizer.step(grads)
    
    return total_loss / n

def evaluate(net, X, y):
    """Evaluate accuracy."""
    logits = net.forward(X)
    preds = np.argmax(logits, axis=1)
    return np.mean(preds == y)

In [ ]:
# Train the network
net = SimpleNetwork([2, 32, 16, 2])
params, _ = net.get_params_and_grads()
optimizer = Adam(params, lr=0.01)

train_losses = []
train_accs = []
test_accs = []

for epoch in range(100):
    loss = train_epoch(net, X_train, y_train, optimizer)
    train_acc = evaluate(net, X_train, y_train)
    test_acc = evaluate(net, X_test, y_test)
    
    train_losses.append(loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch}: Loss={loss:.4f}, Train Acc={train_acc:.3f}, Test Acc={test_acc:.3f}")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')

ax2.plot(train_accs, label='Train')
ax2.plot(test_accs, label='Test')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy')
ax2.legend()

plt.tight_layout()
plt.show()

## Summary

In this notebook, we:
1. Implemented loss functions (MSE, cross-entropy)
2. Built layers with forward and backward passes
3. Implemented SGD, Momentum, and Adam optimizers
4. Added L2 regularization and dropout
5. Created a complete training loop

**Next:** Module 3 covers Convolutional Neural Networks.